# B3.13 Update

From April 14 2025 -- now (May 9 2025)

Check metadata for old sequences to see if anything has been added

In [ ]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = home + "GISAID_Andersen_Combined_Files/04-14-2025--05-09-2025_B3_13/"
downloads = home + "Andersen_Downloads/"
temp_files = home + "Andersen_Temp_Files/"
complete_files = home + "Andersen_Complete_Files/"

update_date = "05-09-2025"

os.chdir(originals)

## Andersen Lab

In [2]:
# Andersen

# Read metadata

metadata_folder = downloads + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# print(len(metadata)) # 7397 rows

# Get rid of missing dates; they won't be counted anyway
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]

# # Find only >= 2024 to start
# metadata["Collection_Date_Compare"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata = metadata[metadata["Collection_Date_Compare"] >= datetime(2021, 11, 1).strftime("%Y-%m-%d")] # Note that those with only years will default to today

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= datetime(2025, 4, 14).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= datetime(2025, 5, 9).strftime("%Y-%m-%d")]

print(len(metadata)) # 289 rows between 4/14/2025 and 5/9/2025

genotypes = ["B3.13"]

289


### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

In [3]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

metadata["Genotype"] = genoflu_results["Genotype"]
metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes

# Get only the genotypes we want: B3.13 

b313_only = genoflu_results[genoflu_results["Genotype"] == "B3.13"]
print(b313_only)
b313_only = b313_only.rename(columns={"sample": "Run"})

metadata = metadata.merge(b313_only, on=["Run"], how="inner")

print(len(metadata)) 

# display(metadata)

           sample                 date       File Name Genotype  \
2     SRR32125669  2025-04-08_15-51-59  SRR32125669.fa    B3.13   
3     SRR31597237  2025-04-08_15-27-07  SRR31597237.fa    B3.13   
4     SRR31605135  2025-04-08_15-30-14  SRR31605135.fa    B3.13   
7     SRR31605198  2025-04-08_15-31-07  SRR31605198.fa    B3.13   
9     SRR31597080  2025-04-08_15-24-54  SRR31597080.fa    B3.13   
...           ...                  ...             ...      ...   
8430  SRR33124636  2025-04-15_06-38-07  SRR33124636.fa    B3.13   
8431  SRR33124637  2025-04-15_06-38-12  SRR33124637.fa    B3.13   
8432  SRR33124638  2025-04-15_06-38-12  SRR33124638.fa    B3.13   
8433  SRR33124639  2025-04-15_06-38-07  SRR33124639.fa    B3.13   
8434  SRR33124640  2025-04-15_06-38-07  SRR33124640.fa    B3.13   

                            Genotype List Used, >=98.0%  \
2     PB2:am2.2, PB1:am4, PA:ea1, HA:ea1, NP:am8, NA...   
3     PB2:am2.2, PB1:am4, PA:ea1, HA:ea1, NP:am8, NA...   
4     PB2:am2.2, P

In [4]:
# # Get specific geolocation from genbank_mapping.tsv

# genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
# genbank_mapping["Run"] = genbank_mapping["sra_run"]
# # genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
# genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# print(genbank_mapping["name_state"])
# print(len(metadata_genbank))
# display(metadata_genbank) # No state information since 3/18/2025?

In [5]:
# If no states

metadata_genbank = metadata

metadata_genbank["name_state"] = "USA"

In [6]:
# # Get all dates
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank))

# # Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank.csv")

In [7]:
# Upload saved data -- if doing this, make sure the above cell is commented out
os.chdir(temp_files + "saved/")
metadata_genbank = pd.read_csv("metadata_genbank_5-9-2025.csv")
os.chdir(temp_files)

In [8]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genbank)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(home)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['cat', 'cattle']
[]
                        avian               cattle        feline  \
0            great_horned_owl            dairy_cow           cat   
1                common_raven               cattle  domestic_cat   
2               cooper's_hawk  cattle milk product     feral_cat   
3                coopers_hawk          bovine_milk        feline   
4                     peafowl              bovine   domestic-cat   
..                        ...                  ...           ...   
392        anas platyrhynchos                  NaN           NaN   
393  haliaeetus leucocephalus                  NaN           NaN   
394        larus occidentalis                  NaN           NaN   
395                 icteridae                  NaN           NaN   
396     lophodytes cucullatus                  NaN           NaN   

      other_mammal       human         other  new  
0       deer mouse  washington         mixed  NaN  
1      house_mouse      nevada   environment  NaN  
2     

In [9]:
# Get animals from animal reference
os.chdir(home)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_genbank, animals_ref) # Get host type

metadata_genbank["years"] = metadata_genbank["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

In [10]:
for num, collection_date in enumerate(metadata_genbank["Collection_Date_Specific"]):
    if collection_date != collection_date: # If nan
        metadata_genbank.loc[num, "Collection_Date_Specific"] = metadata_genbank.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata_genbank.loc[num, "Collection_Date_Specific"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata_genbank.loc[num, "Collection_Date_Specific"] = date

# Make names

names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Collection_Date_Specific"].apply(lambda x: str(x)) + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype_y"] # "|B3.13"

metadata_genbank["Name"] = names

os.chdir(complete_files)

metadata_genbank.to_csv("metadata_genbank_B3_13_" + update_date + ".csv")

# display(metadata_genbank)

In [11]:
# Get information to create the fasta files

fasta_folder = downloads + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype_y"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [12]:
# Create fasta files 

os.chdir(complete_files + "2025-04-14--2025-05-09_B3_13/")

for pair in fasta_files.keys():
    output_path = complete_files + "2025-04-14--2025-05-09_B3_13/" + pair + "_andersen_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

>A/CATTLE/USA/25-006243-005/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/USA/25-006243-002/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/USA/25-006243-001/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/USA/25-006240-005/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/USA/25-006031-002/2025|H5N1|2025|cattle|B3.13
>A/CAT/USA/25-007097-002/2025|H5N1|2025-02-22|feline|B3.13
>A/CAT/USA/25-007097-001/2025|H5N1|2025-02-22|feline|B3.13
>A/CAT/USA/25-006544-001/2025|H5N1|2025-02-19|feline|B3.13
>A/CAT/USA/25-006543-001/2025|H5N1|2025|feline|B3.13
>A/CATTLE/USA/25-006954-002/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/USA/25-006508-014/2025|H5N1|2024-11-19|cattle|B3.13
>A/CATTLE/USA/25-006029-002/2025|H5N1|2025-02-12|cattle|B3.13
>A/CATTLE/USA/25-006508-013/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/USA/25-006508-004/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/USA/25-006506-002/2025|H5N1|2024-11-29|cattle|B3.13
>A/CATTLE/USA/25-006276-004/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/USA/25-006276-003/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/USA/25-006276-

## GISAID

In [13]:
# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
complete_files = home + "GISAID_Complete_Fasta_Files/04-14-2025--05-09-2025_B3_13_northa_southa_ant/"

update_date = "05-09-2025"

os.chdir(originals)

In [14]:
# Have user type in username and password

username = input("Username: ")
password = input("Password: ")
browser = input("Browser: ")
sleep_time = input("Seconds to sleep in between clicks: ")
start_date = input("Start date (format: YYYY-MM-DD): ")
end_date = input("End date (format: YYYY-MM-DD): ")

genotypes = ["B3.13"]

In [ ]:
# open_gisaid(username, password, browser, sleep_time, start_date, end_date)

1


In [16]:
all_metadata_files = []
all_fasta_files = []

# Grab files
for dirpath, dirs, files in os.walk(downloads):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name, engine="xlrd")
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)

unique_animals_all = [] # find unique animals to sort them later

os.chdir(home)

# Separate fastas by segment
segment_fastas = []
unique_segments = []
for i, fasta in enumerate(all_fasta_files):

    metadata = all_metadata_files[i]

    # print(fasta.loc[i, "Isolate_Name"])
    
    unique_animals = sort_animals(fasta) # Find unique animals
    # print("Animals: ", unique_animals)
    unique_animals_all.append(unique_animals)

    animals_ref = pd.read_csv("animals_ref.csv")

    fastas, unique_segments = separate_fasta_by_seg(metadata, fasta, animals_ref, genotypes) # Separate the fasta dataframes into 8 different files based on segment

    segment_fastas.append(fastas)

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)


c:\Users\maksiaevai.NCBI_NT\Documents\Avian_Flu\utils.py:459: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
c:\Users\maksiaevai.NCBI_NT\Documents\Avian_Flu\utils.py:463: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name
c:\Users\maksiaevai.NCBI_NT\Documents\Avian_Flu\utils.py:459: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in

In [19]:
huge_fasta = pd.DataFrame()

for fastas in segment_fastas: 
    # print(len(fastas))
    # break
    for f in fastas: 
        # print(f)
        # break 
        huge_fasta = pd.concat([huge_fasta, f])

# print(huge_fasta.columns)

# Now separate huge_fasta into 16 fastas
big_fastas = []

# print(huge_fasta)

# genotypes = ["B3.13", "D1.1"] # , "D1.3"]
for gen in genotypes:
    # print(gen)
    big_fasta = huge_fasta[huge_fasta["Genotype"] == gen]
    # print(gen)
    for seg in unique_segments:
        seg_specific_fasta = big_fasta[big_fasta["Segment"] == seg]
        big_fastas.append(seg_specific_fasta)


# Now that we have 16 fastas, write the files
for fasta in big_fastas:

    # Create a dictionary to create a file
    fasta_df = fasta[["New_Name", "Sequence"]]
    fasta_dict = pd.Series(fasta_df.Sequence.values,index=fasta_df.New_Name).to_dict()
    # print(fasta["Genotype"])
    # Create fasta file 
    try: 
        output_path = complete_files + fasta["Genotype"].values[0] + "_" + fasta["Segment"].values[0] + "_GISAID_" + end_date + ".fasta" # Genotype and Segment should all be the same
        output_file = open(output_path, "w")
        for item in fasta_dict.keys():
            # print(item)
            value = fasta_dict[item] + "\n"
            # print(value)
            output_file.write(item)
            output_file.write(value)
        print("Succeeded in finding results for genotype: ", fasta["Genotype"].values[0])
        output_file.close()
    except:
        # print(fasta["Genotype"])
        # print(fasta)
        print("Could not find any results for genotype.")
        # continue

Succeeded in finding results for genotype:  B3.13
Succeeded in finding results for genotype:  B3.13
Succeeded in finding results for genotype:  B3.13
Succeeded in finding results for genotype:  B3.13
Succeeded in finding results for genotype:  B3.13
Succeeded in finding results for genotype:  B3.13
Succeeded in finding results for genotype:  B3.13
Succeeded in finding results for genotype:  B3.13


In [20]:
# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["avian", "cattle", "feline", "other_mammal", "human", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(home)
animals_df.to_csv("animals_ref_to_sort.csv")

['dunlin', 'barn_owl', 'hooded_merganser', 'american_crow', 'turkey_vulture', 'rough-legged_hawk', 'mallard-black_duck_hybrid', 'sandhill_crane', 'western_sandpiper', 'great_egret', 'peregrine', 'american_black_duck', "cooper's_hawk", 'mute_swan', 'rock_goose', 'red-shouldered_hawk', 'red_fox', 'brown_skua', 'guineafowl', 'skunk', 'great_black-backed_gull', 'vulture', 'chukar', 'great_horned_owl', 'mallard', 'snow_goose', 'duck', 'flamingo', 'canada_goose', 'herring_gull', 'cat', 'sand_crane', 'crow', 'bald_eagle', 'swan', 'bufflehead', 'red-tailed_hawk', 'snowy_owl', "bonaparte's_gull", 'goose', 'turkey', "ross's_goose", 'environment', 'western_gull', 'chicken', 'dairy_cow', 'black_vulture']
47
[]
                        avian               cattle        feline  \
0            great_horned_owl            dairy_cow           cat   
1                common_raven               cattle  domestic_cat   
2               cooper's_hawk  cattle milk product     feral_cat   
3                coo

## De-Duplication

In [21]:
# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
combined_files = home + "GISAID_Andersen_Combined_Files/04-14-2025--05-09-2025_B3_13/"
downloads = home + "Andersen_Downloads/"
temp_files = home + "Andersen_Temp_Files/"
complete_files = home + "Andersen_Complete_Files/"

update_date = "05-09-2025"

gisaid = home + "GISAID_Complete_Fasta_Files/04-14-2025--05-09-2025_B3_13_northa_southa_ant/"

os.chdir(gisaid)

dfs_gisaid = create_dataframes(gisaid)

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2


In [22]:
dfs_andersen = create_dataframes(complete_files + "2025-04-14--2025-05-09_B3_13/")

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2


In [23]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
defaultdict(<class 'list'>, {'B3.13_HA': [   isolate_partial                                        full_header  \
0       006243-005  >A/CATTLE/USA/25-006243-005/2025|H5N1|2025|cat...   
1       006243-002  >A/CATTLE/USA/25-006243-002/2025|H5N1|2025|cat...   
2       006243-001  >A/CATTLE/USA/25-006243-001/2025|H5N1|2025|cat...   
3       006240-005  >A/CATTLE/USA/25-006240-005/2025|H5N1|2025|cat...   
4       006031-002  >A/CATTLE/USA/25-006031-002/2025|H5N1|2025|cat...   
5       007097-002  >A/CAT/USA/25-007097-002/2025|H5N1|2025-02-22|...   
6       007097-001  >A/CAT/USA/25-007097-001/2025|H5N1|2025-02-22|...   
7       006544-001  >A/CAT/USA/25-006544-001/2025|H5N1|2025-02-19|...   
8       006543-001  >A/CAT/USA/25-006543-001/2025|H5N1|2025|feline...   
9       006954-002  >A/CATTLE/USA/25-006954-002/2025|H5N1|2025|cat...   
10      006508-014  >A/CATTLE/USA/25-006508-014/2025|H5N1|2024-11-...   
11      

In [24]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                gisaid_df = dfs_gisaid[gisaid_key][0]
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                full_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")
                full_dfs[andersen_key].append(full_df)

print(full_dfs)

defaultdict(<class 'list'>, {'B3.13_HA': [    isolate_partial                                        full_header  \
1        006243-002  >A/CATTLE/USA/25-006243-002/2025|H5N1|2025|cat...   
3        006240-005  >A/CATTLE/USA/25-006240-005/2025|H5N1|2025|cat...   
4        006031-002  >A/CATTLE/USA/25-006031-002/2025|H5N1|2025|cat...   
9        006954-002  >A/CATTLE/USA/25-006954-002/2025|H5N1|2025|cat...   
10       006508-014  >A/CATTLE/USA/25-006508-014/2025|H5N1|2024-11-...   
..              ...                                                ...   
153      000604-001  >A/dairy_cow/California/25_000604-001/2024|H5N...   
154      004923-002  >A/dairy_cow/California/25_004923-002/2024|H5N...   
155      000610-001  >A/dairy_cow/California/25_000610-001/2024|H5N...   
156      034835-001  >A/dairy_cow/California/24_034835-001-R2/2024|...   
157      004923-001  >A/dairy_cow/California/25_004923-001/2024|H5N...   

                                              sequence  
1    ATGAAGA

In [25]:
# Create FASTA files per segment

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + update_date + "_B3_13.fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
